In [9]:
# [IMPORT] Machine learning libraries
import numpy as np
import pandas as pd

# Load raw extracted tmdb movies dataset
df=pd.read_csv('tmdb_movies_dataset.csv')

In [10]:
# df.info()
df.head()
# df.describe()
# df.isna().sum()

,id,title,original_title,overview,release_date,popularity,vote_average,vote_count,genre_ids,original_language
0,1087192,How to Train Your Dragon,How to Train Your Dragon,"On the rugged isle of Berk, where Vikings and ...",2025-06-06,1075.8358,8.126,1013,"[14, 10751, 28]",en
1,1071585,M3GAN 2.0,M3GAN 2.0,After the underlying tech for M3GAN is stolen ...,2025-06-25,828.4871,7.711,376,"[28, 878, 53]",en
2,1315986,Man with No Past,Man with No Past,"Waking up in an unfamiliar city, a man with no...",2025-01-13,510.3027,6.775,20,"[28, 18]",en
3,1011477,Karate Kid: Legends,Karate Kid: Legends,"After a family tragedy, kung fu prodigy Li Fon...",2025-05-08,368.2810,7.254,478,"[28, 12, 18]",en
4,1061474,Superman,Superman,"Superman, a journalist in Metropolis, embarks ...",2025-07-09,359.1730,7.500,961,"[878, 12, 28]",en


In [11]:
import ast

def preprocess_data(df):    
    # Imputation
    df['overview']=df['overview'].fillna('Unknown')

    # convert to DateTime
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    df['release_date']=df['release_date'].fillna(df['release_date'].median())
    # split 'release_date' into 'year', 'month', 'day'
    df['release_year'] = df['release_date'].dt.year.astype(int)
    df['release_month'] = df['release_date'].dt.month.astype(int)
    df['release_day'] = df['release_date'].dt.day.astype(int)

    # Drop columns
    df=df.drop(columns=['id', 'title', 'original_title', 'release_date'])

    return df

def feature_engineer(df):
    # Convert genre_ids from string to list if needed
    df['genre_ids']=df['genre_ids'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    return df

df=preprocess_data(df)
df=feature_engineer(df)

In [12]:
df.head()

,overview,popularity,vote_average,vote_count,genre_ids,original_language,release_year,release_month,release_day
0,"On the rugged isle of Berk, where Vikings and ...",1075.8358,8.126,1013,"[14, 10751, 28]",en,2025,6,6
1,After the underlying tech for M3GAN is stolen ...,828.4871,7.711,376,"[28, 878, 53]",en,2025,6,25
2,"Waking up in an unfamiliar city, a man with no...",510.3027,6.775,20,"[28, 18]",en,2025,1,13
3,"After a family tragedy, kung fu prodigy Li Fon...",368.2810,7.254,478,"[28, 12, 18]",en,2025,5,8
4,"Superman, a journalist in Metropolis, embarks ...",359.1730,7.500,961,"[878, 12, 28]",en,2025,7,9


In [13]:
numerical_cols=['popularity', 'vote_average', 'vote_count']
categorical_cols=['original_language', 'release_year', 'release_month', 'release_day']
textual_cols=['overview']

In [14]:
from scipy.stats import skew

print("[ BEFORE TRANSFORMATION ]")
for col in numerical_cols:
    print(f"Skewness[{col}]: {df[col].skew():.2f}")
    
def winsorize_series(series, lower_quantile=0.01, upper_quantile=0.99):
    lower = series.quantile(lower_quantile)
    upper = series.quantile(upper_quantile)
    return series.clip(lower, upper)

def transform(cols):
    for col in cols:
        skewness=df[col].skew()
        if skewness > 0.75: # log transform
            df[col]=np.log1p(df[col])
        elif skewness < -0.5: # reflect, then log transform
            df[col] = df[col].max() + 1 - df[col]
            df[col] = np.log1p(df[col])
        else: # sqrt transform
            df[col]=np.sqrt(df[col])

print("[ AFTER TRANSFORMATION ]")
df['popularity']=winsorize_series(df['popularity'])
transform(['popularity', 'vote_average', 'vote_count'])

for col in numerical_cols:
    print(f"Skewness[{col}]: {df[col].skew():.2f}")

[ BEFORE TRANSFORMATION ]
Skewness[popularity]: 42.32
Skewness[vote_average]: -1.09
Skewness[vote_count]: 5.18
[ AFTER TRANSFORMATION ]
Skewness[popularity]: 0.48
Skewness[vote_average]: 0.25
Skewness[vote_count]: -0.44


In [15]:
import matplotlib.pyplot as plt
import seaborn as sns

# corr_matrix=df.corr(numeric_only=True)
# plt.figure(figsize=(10, 7))
# sns.heatmap(corr_matrix, 
#             annot=True,          # Show correlation values
#             cmap='coolwarm',     # Color scheme (red-blue diverging)
#             vmin=-1, vmax=1,     # Correlation range
#             center=0,            # Center point for diverging colormap
#             square=True,         # Make cells square
#             fmt='.2f',           # Format numbers to 2 decimal places
#             annot_kws={'size': 10},  # Font size for annotations
#             linewidths=0.5)      # Lines between cells
# plt.title('Feature Correlation Heatmap')
# plt.show()

In [16]:
from collections import Counter
# Raw KNN Algorithm
class KNN:
    def __init__(self, k=5, weighted=False):
        self.k = k
        self.weighted = weighted

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def _euclidean(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def _get_neighbors(self, x):
        distances = []

        for i, x_train in enumerate(self.X_train):
            dist = self._euclidean(x, x_train)
            distances.append((dist, self.y_train[i]))

        # Sort by distance
        distances.sort(key=lambda x: x[0])

        return distances[:self.k]

    def predict(self, X):
        X = np.array(X)
        predictions = []

        for x in X:
            neighbors = self._get_neighbors(x)

            # Classification
            if isinstance(neighbors[0][1], str) or isinstance(neighbors[0][1], int):
                predictions.append(self._predict_class(neighbors))
            else:
                predictions.append(self._predict_regression(neighbors))

        return np.array(predictions)

    def _predict_class(self, neighbors):
        if not self.weighted:
            labels = [label for _, label in neighbors]
            return Counter(labels).most_common(1)[0][0]

        # Weighted classification: weight = 1 / distance
        weights = {}
        for dist, label in neighbors:
            w = 1 / (dist + 1e-8)
            weights[label] = weights.get(label, 0) + w

        return max(weights, key=weights.get)

    def _predict_regression(self, neighbors):
        if not self.weighted:
            values = [label for _, label in neighbors]
            return np.mean(values)

        # Weighted regression
        weighted_sum = 0
        total_weight = 0
        for dist, value in neighbors:
            w = 1 / (dist + 1e-8)
            weighted_sum += w * value
            total_weight += w

        return weighted_sum / total_weight

In [17]:
from heapq import heappush, heappop

class KDNode:
    def __init__(self, point, label, axis, left=None, right=None):
        self.point = point        # The feature vector
        self.label = label        # Class label or value
        self.axis = axis          # Split dimension
        self.left = left          # Left subtree
        self.right = right        # Right subtree

def build_kdtree(points, labels, depth=0):
    if len(points) == 0:
        return None

    k = points.shape[1]               # Number of dimensions
    axis = depth % k                  # Choose split dimension

    # Sort and choose median
    sorted_idx = points[:, axis].argsort()
    points = points[sorted_idx]
    labels = labels[sorted_idx]

    median = len(points) // 2

    return KDNode(
        point=points[median],
        label=labels[median],
        axis=axis,
        left=build_kdtree(points[:median], labels[:median], depth + 1),
        right=build_kdtree(points[median + 1:], labels[median + 1:], depth + 1)
    )

def kdtree_knn_search(node, target, k, heap):
    if node is None:
        return

    # Compute distance to current node
    dist = np.linalg.norm(target - node.point)

    # Max heap: store (-dist, label)
    heappush(heap, (-dist, node.label))
    if len(heap) > k:
        heappop(heap)

    axis = node.axis

    # Choose branch to explore first
    if target[axis] < node.point[axis]:
        near_branch = node.left
        far_branch = node.right
    else:
        near_branch = node.right
        far_branch = node.left

    # Explore near branch
    kdtree_knn_search(near_branch, target, k, heap)

    # Check if we need to explore far branch
    if len(heap) < k or abs(target[axis] - node.point[axis]) < -heap[0][0]:
        kdtree_knn_search(far_branch, target, k, heap)


In [18]:
# KD-Tree Optimized KNN
class KNN_KDTree:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self.tree = build_kdtree(X, y)
        self.y_type = type(y[0])  # detect classification or regression

    def predict_one(self, x):
        heap = []
        kdtree_knn_search(self.tree, np.array(x), self.k, heap)

        # Extract neighbors (distance, label)
        neighbors = [(-dist, label) for dist, label in heap]

        # Classification
        if self.y_type in [int, str]:
            labels = [label for dist, label in neighbors]
            return Counter(labels).most_common(1)[0][0]

        # Regression
        distances = [dist for dist, label in neighbors]
        labels = [label for dist, label in neighbors]
        return np.average(labels, weights=[1/(d+1e-8) for d in distances])

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

In [19]:
import time
start = time.time()
...
end = time.time()